In [ ]:
import numpy as np
from qutip import *
from scipy.linalg import sqrtm, eigvalsh
from scipy.stats import linregress
from numba import njit, prange
import pickle
import os
import gc

In [ ]:
%matplotlib ipympl
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from IPython.display import Image, display, Math

In [ ]:
@njit(parallel=True)
def compute_pauli_expectations_all_trajectories(pop_00, pop_11, coh_01):
    """
    Computes the expectation values of sigma_x, sigma_y, and sigma_z 
    for ALL individual trajectories.
    
    Optimized math:
    <sigma_z> = rho_00 - rho_11
    <sigma_x> = 2 * Re(rho_01)
    <sigma_y> = -2 * Im(rho_01)
    
    Inputs:
        2D NumPy arrays: (time_steps, N_traj) 
        
    Returns:
        sigma_x_matrix, sigma_y_matrix, sigma_z_matrix: 2D arrays (time_steps, N_traj)
    """
    time_steps = pop_00.shape[0]
    N_traj = pop_00.shape[1]
    
    # Pre-allocate output 2D arrays (time_steps, N_traj)
    sigma_x_matrix = np.zeros((time_steps, N_traj))
    sigma_y_matrix = np.zeros((time_steps, N_traj))
    sigma_z_matrix = np.zeros((time_steps, N_traj))
    
    # Outer loop over time
    for t in range(time_steps):
        # Parallel loop over all trajectories (using all CPU cores)
        for n in prange(N_traj):
            
            # <sigma_z> = population difference
            sigma_z_matrix[t, n] = pop_00[t, n] - pop_11[t, n]
            
            # <sigma_x> = 2 * Real part of coherence
            sigma_x_matrix[t, n] = 2.0 * np.real(coh_01[t, n])
            
            # <sigma_y> = -2 * Imaginary part of coherence
            sigma_y_matrix[t, n] = -2.0 * np.imag(coh_01[t, n])
            
    return sigma_x_matrix, sigma_y_matrix, sigma_z_matrix

## Results Analysis

In [ ]:
# ====================================
# Physical & Simulation Parameters
# ====================================

# Time parameters
dt = 0.01
tf = 100.0
time_steps = int(tf / dt)

# List of Number of trajectories to analyze
N_traj_list = [100, 200, 300, 400, 500, 600, 700, 800, 900, 1000, 2000, 3000, 4000, 5000, 6000, 7000, 8000, 9000, 10000, 20000]        

In [ ]:
# ============================================================
# GLOBAL CONFIGURATION :    
#   'QJ' → Quantum Jump
#   'SD' → State Diffusion
# ============================================================


MODE = 'SD'   # Switch this to 'SD' 

# Configuration mapping based on MODE
_cfg = {
    'QJ': {
        'Input_file': "../Results/Data/Complete_rho/pop_1.0_result_mode_QJ_dt0p010000_Ntraj20000.npz",
        'Output_dir': "../Results/Plot/Sxyz_Analysis/Avg/QJ" 
    },
    'SD': {
        'Input_file': "../Results/Data/Complete_rho/pop_1.0_result_mode_SD_dt0p010000_Ntraj20000.npz",
        'Output_dir': "../Results/Plot/Sxyz_Analysis/Avg/SD"
    },
}

# Apply the selected configuration
cfg = _cfg[MODE]

# Set the global Input and Output_dir dynamically based on the current mode
Input_file = cfg['Input_file']
Output_dir = cfg['Output_dir']

# Create the output directory if it doesn't exist
os.makedirs(Output_dir, exist_ok=True)

print(f"--- Configuration Setup ---")
print(f"Current mode : {MODE}")
print(f"Input Data   : {cfg['Input_file']}")
print(f"Output Plots : {Output_dir}")

In [ ]:
# ===========================
# General Setup for Plotting
# ===========================

# Global Style Settings (Matplotlib rcParams)
plt.rcParams.update({
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 11,
    'ytick.labelsize': 11,
    'legend.fontsize': 10,
    'figure.figsize': (10, 5),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.linestyle': ':',
    'figure.autolayout': True # plt.tight_layout()
})

# Automatic Figure Saving Function
def save_fig(fig, filename):
    """
    Saves the figure in both PNG or PDF
    """
    path_png = os.path.join(Output_dir, f"{filename}.png")
    # path_pdf = os.path.join(Output_dir, f"{filename}.pdf")  # save in pdf
    
    fig.savefig(path_png, dpi=300, bbox_inches='tight')
    # fig.savefig(path_pdf, bbox_inches='tight') # save in pdf
    print(f"Figure saved in: {Output_dir}/{filename}")

### Data Extraction

In [ ]:
# ------------
# Load Results
# ------------
data = np.load(Input_file)

times = data['times']
time_steps = len(times)

# ------------
# Extract data
# ------------

# Extract Lindblad populations
rho_lindblad = data['rho_list_lindblad']
# lindblad_00 = np.real(rho_lindblad[:, 0, 0])
# lindblad_11 = np.real(rho_lindblad[:, 1, 1])
# lindblad_01 = rho_lindblad[:, 0, 1]

# Extract Trajectories data 
pop_00 = data['pop_00']
pop_11 = data['pop_11']
coh_01 = data['coh_01']
coh_10 = np.conj(coh_01)

print("--- Data Loading Completed ---")

# ==============================
# Density Matrix reconstruction
# ==============================

N_traj_tot = pop_00.shape[1]

# rho_complete = np.zeros((time_steps, N_traj_tot, 2, 2), dtype=np.complex128)
# rho_complete[:, :, 0, 0] = pop_00
# rho_complete[:, :, 1, 1] = pop_11
# rho_complete[:, :, 0, 1] = coh_01
# rho_complete[:, :, 1, 0] = np.conj(coh_01)


### Pauli Expectation Value

In [ ]:

# =================
# PAULI OBSERVABLES 
# =================

# ---------
# Lindblad
# ---------

lindblad_sx = np.real(rho_lindblad[:, 0, 1] + rho_lindblad[:, 1, 0])
lindblad_sy = np.real(1j * (rho_lindblad[:, 0, 1] - rho_lindblad[:, 1, 0])) 
lindblad_sz = np.real(rho_lindblad[:, 0, 0] - rho_lindblad[:, 1, 1])

# -------------
# Trajectories
# -------------
print("Starting Pauli expectation calculation for all trajectories...")

# Call the optimized Numba function directly with the 2D arrays
all_sigma_x, all_sigma_y, all_sigma_z = compute_pauli_expectations_all_trajectories(
    pop_00, pop_11, coh_01
)

print("Calculation completed!")
print(f"Shape of sigma_z matrix: {all_sigma_z.shape}")

In [ ]:
# ====================================================================
# ERROR CALCULATION: PAULI OBSERVABLES VS LINDBLAD
# ====================================================================

print("Starting error calculation for Sx, Sy, Sz...")

# Initialize simple lists to store the time-averaged and maximum ERROR for each N
mean_err_sx_results = []
max_err_sx_results  = []

mean_err_sy_results = []
max_err_sy_results  = []

mean_err_sz_results = []
max_err_sz_results  = []

# Loop over the number of trajectories defined in N_traj_list
for N in N_traj_list:
    
    # 1. Calculate the ensemble average over N trajectories for each time step
    sx_ensemble_mean = np.mean(all_sigma_x[:, :N], axis=1)
    sy_ensemble_mean = np.mean(all_sigma_y[:, :N], axis=1)
    sz_ensemble_mean = np.mean(all_sigma_z[:, :N], axis=1)
    
    # 2. Calculate the absolute error w.r.t. the exact Lindblad solution
    err_sx = np.abs(sx_ensemble_mean - lindblad_sx)
    err_sy = np.abs(sy_ensemble_mean - lindblad_sy)
    err_sz = np.abs(sz_ensemble_mean - lindblad_sz)
    
    # 3. Calculate the time-averaged value of the error and append to lists
    mean_err_sx_results.append(np.mean(err_sx))
    mean_err_sy_results.append(np.mean(err_sy))
    mean_err_sz_results.append(np.mean(err_sz))
    
    # 4. Calculate the maximum value over time of the error and append to lists
    max_err_sx_results.append(np.max(err_sx))
    max_err_sy_results.append(np.max(err_sy))
    max_err_sz_results.append(np.max(err_sz))

# Optional: Convert lists to NumPy arrays for easier plotting later
mean_err_sx_results = np.array(mean_err_sx_results)
max_err_sx_results  = np.array(max_err_sx_results)
mean_err_sy_results = np.array(mean_err_sy_results)
max_err_sy_results  = np.array(max_err_sy_results)
mean_err_sz_results = np.array(mean_err_sz_results)
max_err_sz_results  = np.array(max_err_sz_results)

print("Error calculation completed successfully!")

## Plot

In [ ]:
base_output_dir = Output_dir  # Store the base output directory for later use

In [ ]:
# ========================================================
# MEAN AND MAX ERROR OF Sz VS N_TRAJECTORIES
# ========================================================

plt.close('all')

# Create a figure with 2 subplots side by side
fig_Sz, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- First Subplot: Mean Error ---
# We can pass the array directly since it matches the length of N_traj_list
ax1.plot(N_traj_list, mean_err_sz_results, marker='o', linestyle='-', color='tab:blue', label=f'mode = {MODE}')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel(r'Number of Trajectories ($N_{\text{traj}}$)', fontsize=12)
ax1.set_ylabel(r'Mean Error of $S_z$', fontsize=12)
ax1.set_title(r'Time-Averaged Error of $S_z$ vs $N_{\text{traj}}$', fontsize=14)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6, which="both")

# --- Second Subplot: Max Error ---
ax2.plot(N_traj_list, max_err_sz_results, marker='^', linestyle='--', color='tab:red', label=f'mode = {MODE}')

ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel(r'Number of Trajectories ($N_{\text{traj}}$)', fontsize=12)
ax2.set_ylabel(r'Max Error of $S_z$', fontsize=12)
ax2.set_title(r'Maximum Error of $S_z$ vs $N_{\text{traj}}$', fontsize=14)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6, which="both")

# Adjust layout to prevent overlapping
plt.tight_layout()

# ---------------------
# FIGURE SAVING FOR Sz
# ---------------------

# Dynamically set the output directory using the base path
Output_dir = os.path.join(base_output_dir, "Sz")

# Check if the directory exists, otherwise create it
if not os.path.exists(Output_dir):
    os.makedirs(Output_dir)
    print(f"Created directory: {Output_dir}")

# Save the figure 
save_fig(fig_Sz, "Mean_Max_Error_Sz")

plt.show()

In [ ]:
# ====================================================================
# SCALING FIT & PLOT: MEAN ERROR OF Sz vs N_TRAJ
# ====================================================================

plt.close('all')

plt.figure(figsize=(10, 6))

print(f"--- Scaling Fit Results: Mean Error of Sz ({MODE}) ---")

# Filter valid points (must be strictly positive for log10)
valid_N = []
valid_err = []

for N, err_val in zip(N_traj_list, mean_err_sz_results):
    if err_val > 0:
        valid_N.append(N)
        valid_err.append(err_val)
        
# Convert lists to arrays for math operations
x_data = np.array(valid_N)
y_data = np.array(valid_err)

# Take logarithms for the log-log fit: log(Error) = slope * log(N) + intercept
log_x = np.log10(x_data)
log_y = np.log10(y_data)

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(log_x, log_y)
print(f"Slope (m):     {slope:+.4f}")
print(f"Intercept (q): {intercept:+.4f}")
print(f"R-squared:     {r_value**2:.4f}")

# Plotting the original data points in log-log space
plt.scatter(log_x, log_y, color='tab:blue', alpha=0.7, marker='o', s=60, label=f'Data ({MODE})')

# Generate the theoretical fit line points
x_fit = np.linspace(min(log_x), max(log_x), 100)
y_fit = slope * x_fit + intercept

# Plot the fit line and include the equation in the legend
plt.plot(x_fit, y_fit, color='tab:red', linewidth=2, 
         label=f'Linear Fit: $y = {slope:.2f}x {intercept:+.2f}$')

# Formatting the plot
plt.xlabel(r"$\log_{10}(N_{\text{traj}})$", fontsize=12)
plt.ylabel(r"$\log_{10}(\langle \text{Err}_{S_z} \rangle_t)$", fontsize=12)
plt.title(f"Scaling Fit: Mean Time-Averaged Error of $S_z$ vs Trajectories ({MODE})", fontsize=14)

# Legend and Grid
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.5, which="both")
plt.tight_layout()

# ---------------------
# FIGURE SAVING FOR Sz
# ---------------------

# Dynamically set the output directory using the base path
Output_dir = os.path.join(base_output_dir, "Sz")

# Check if the directory exists, otherwise create it
if not os.path.exists(Output_dir):
    os.makedirs(Output_dir)
    print(f"Created directory: {Output_dir}")

# Save the figure using the global function
save_fig(plt.gcf(), f"Scaling_Fit_Error_Sz_{MODE}")

plt.show()

In [ ]:
# ========================================================
# MEAN AND MAX ERROR OF Sx VS N_TRAJECTORIES
# ========================================================

plt.close('all')

# Create a figure with 2 subplots side by side
fig_Sx, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- First Subplot: Mean Error ---
# Plotting the time-averaged error for Sx
ax1.plot(N_traj_list, mean_err_sx_results, marker='o', linestyle='-', color='tab:blue', label=f'mode = {MODE}')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel(r'Number of Trajectories ($N_{\text{traj}}$)', fontsize=12)
ax1.set_ylabel(r'Mean Error of $S_x$', fontsize=12)
ax1.set_title(r'Time-Averaged Error of $S_x$ vs $N_{\text{traj}}$', fontsize=14)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6, which="both")

# --- Second Subplot: Max Error ---
# Plotting the maximum error over time for Sx
ax2.plot(N_traj_list, max_err_sx_results, marker='^', linestyle='--', color='tab:red', label=f'mode = {MODE}')

ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel(r'Number of Trajectories ($N_{\text{traj}}$)', fontsize=12)
ax2.set_ylabel(r'Max Error of $S_x$', fontsize=12)
ax2.set_title(r'Maximum Error of $S_x$ vs $N_{\text{traj}}$', fontsize=14)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6, which="both")

# Adjust layout to prevent overlapping labels
plt.tight_layout()
# ---------------------
# FIGURE SAVING FOR Sx
# ---------------------

# Dynamically set the output directory using the base path
Output_dir = os.path.join(base_output_dir, "Sx")

# Check if the directory exists, otherwise create it
if not os.path.exists(Output_dir):
    os.makedirs(Output_dir)
    print(f"Created directory: {Output_dir}")

# Save the figure 
save_fig(fig_Sx, "Mean_Max_Error_Sx")

plt.show()

In [ ]:
# ====================================================================
# SCALING FIT & PLOT: MEAN ERROR OF Sx vs N_TRAJ
# ====================================================================

plt.close('all')

plt.figure(figsize=(10, 6))

print(f"--- Scaling Fit Results: Mean Error of Sx ({MODE}) ---")

# Filter valid points (must be strictly positive for log10)
valid_N = []
valid_err = []

for N, err_val in zip(N_traj_list, mean_err_sx_results):
    if err_val > 0:
        valid_N.append(N)
        valid_err.append(err_val)
        
# Convert lists to arrays for math operations
x_data = np.array(valid_N)
y_data = np.array(valid_err)

# Take logarithms for the log-log fit: log(Error) = slope * log(N) + intercept
log_x = np.log10(x_data)
log_y = np.log10(y_data)

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(log_x, log_y)
print(f"Slope (m):     {slope:+.4f}")
print(f"Intercept (q): {intercept:+.4f}")
print(f"R-squared:     {r_value**2:.4f}")

# Plotting the original data points in log-log space
plt.scatter(log_x, log_y, color='tab:blue', alpha=0.7, marker='o', s=60, label=f'Data ({MODE})')

# Generate the theoretical fit line points
x_fit = np.linspace(min(log_x), max(log_x), 100)
y_fit = slope * x_fit + intercept

# Plot the fit line and include the equation in the legend
plt.plot(x_fit, y_fit, color='tab:red', linewidth=2, 
         label=f'Linear Fit: $y = {slope:.2f}x {intercept:+.2f}$')

# Formatting the plot
plt.xlabel(r"$\log_{10}(N_{\text{traj}})$", fontsize=12)
plt.ylabel(r"$\log_{10}(\langle \text{Err}_{S_x} \rangle_t)$", fontsize=12)
plt.title(f"Scaling Fit: Mean Time-Averaged Error of $S_x$ vs Trajectories ({MODE})", fontsize=14)

# Legend and Grid
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.5, which="both")
plt.tight_layout()

# ---------------------
# FIGURE SAVING FOR Sx
# ---------------------

# Dynamically set the output directory using the base path
Output_dir = os.path.join(base_output_dir, "Sx")

# Check if the directory exists, otherwise create it
if not os.path.exists(Output_dir):
    os.makedirs(Output_dir)
    print(f"Created directory: {Output_dir}")

# Save the figure using the global function
save_fig(plt.gcf(), f"Scaling_Fit_Error_Sx_{MODE}")

plt.show()

In [ ]:
# ========================================================
# MEAN AND MAX ERROR OF Sy VS N_TRAJECTORIES
# ========================================================

plt.close('all')

# Create a figure with 2 subplots side by side
fig_Sy, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# --- First Subplot: Mean Error ---
# Plotting the time-averaged error for Sy
ax1.plot(N_traj_list, mean_err_sy_results, marker='o', linestyle='-', color='tab:blue', label=f'mode = {MODE}')

ax1.set_xscale('log')
ax1.set_yscale('log')
ax1.set_xlabel(r'Number of Trajectories ($N_{\text{traj}}$)', fontsize=12)
ax1.set_ylabel(r'Mean Error of $S_y$', fontsize=12)
ax1.set_title(r'Time-Averaged Error of $S_y$ vs $N_{\text{traj}}$', fontsize=14)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.6, which="both")

# --- Second Subplot: Max Error ---
# Plotting the maximum error over time for Sy
ax2.plot(N_traj_list, max_err_sy_results, marker='^', linestyle='--', color='tab:red', label=f'mode = {MODE}')

ax2.set_xscale('log')
ax2.set_yscale('log')
ax2.set_xlabel(r'Number of Trajectories ($N_{\text{traj}}$)', fontsize=12)
ax2.set_ylabel(r'Max Error of $S_y$', fontsize=12)
ax2.set_title(r'Maximum Error of $S_y$ vs $N_{\text{traj}}$', fontsize=14)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.6, which="both")

# Adjust layout to prevent overlapping labels
plt.tight_layout()
# ---------------------
# FIGURE SAVING FOR Sy
# ---------------------

# Dynamically set the output directory using the base path
Output_dir = os.path.join(base_output_dir, "Sy")

# Check if the directory exists, otherwise create it
if not os.path.exists(Output_dir):
    os.makedirs(Output_dir)
    print(f"Created directory: {Output_dir}")

# Save the figure 
save_fig(fig_Sy, "Mean_Max_Error_Sy")

plt.show()

In [ ]:
# ====================================================================
# SCALING FIT & PLOT: MEAN ERROR OF Sy vs N_TRAJ
# ====================================================================

plt.close('all')

plt.figure(figsize=(10, 6))

print(f"--- Scaling Fit Results: Mean Error of Sy ({MODE}) ---")

# Filter valid points (must be strictly positive for log10)
valid_N = []
valid_err = []

for N, err_val in zip(N_traj_list, mean_err_sy_results):
    if err_val > 0:
        valid_N.append(N)
        valid_err.append(err_val)
        
# Convert lists to arrays for math operations
x_data = np.array(valid_N)
y_data = np.array(valid_err)

# Take logarithms for the log-log fit: log(Error) = slope * log(N) + intercept
log_x = np.log10(x_data)
log_y = np.log10(y_data)

# Perform linear regression
slope, intercept, r_value, p_value, std_err = linregress(log_x, log_y)
print(f"Slope (m):     {slope:+.4f}")
print(f"Intercept (q): {intercept:+.4f}")
print(f"R-squared:     {r_value**2:.4f}")

# Plotting the original data points in log-log space
plt.scatter(log_x, log_y, color='tab:blue', alpha=0.7, marker='o', s=60, label=f'Data ({MODE})')

# Generate the theoretical fit line points
x_fit = np.linspace(min(log_x), max(log_x), 100)
y_fit = slope * x_fit + intercept

# Plot the fit line and include the equation in the legend
plt.plot(x_fit, y_fit, color='tab:red', linewidth=2, 
         label=f'Linear Fit: $y = {slope:.2f}x {intercept:+.2f}$')

# Formatting the plot
plt.xlabel(r"$\log_{10}(N_{\text{traj}})$", fontsize=12)
plt.ylabel(r"$\log_{10}(\langle \text{Err}_{S_y} \rangle_t)$", fontsize=12)
plt.title(f"Scaling Fit: Mean Time-Averaged Error of $S_y$ vs Trajectories ({MODE})", fontsize=14)

# Legend and Grid
plt.legend(loc='best')
plt.grid(True, linestyle='--', alpha=0.5, which="both")
plt.tight_layout()

# ---------------------
# FIGURE SAVING FOR Sy
# ---------------------

# Dynamically set the output directory using the base path
Output_dir = os.path.join(base_output_dir, "Sy")

# Check if the directory exists, otherwise create it
if not os.path.exists(Output_dir):
    os.makedirs(Output_dir)
    print(f"Created directory: {Output_dir}")

# Save the figure using the global function
save_fig(plt.gcf(), f"Scaling_Fit_Error_Sy_{MODE}")

plt.show()